# Train a voltage model

Each row in `data.csv` is one measured spectrum and the voltages used to produce it.
Put the spectrum columns first, on the same wavelength grid for every row.
Put the voltage columns after them, in the order you want the model to return them.
Use a header row and keep measurement metadata in a separate file.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

DATA = Path("data.csv")
SPECTRUM_POINTS = 5001
MODEL = "niu"  # "niu" or "convnet"
EPOCHS = 100
BATCH_SIZE = 32
SEED = 0
CHECKPOINT = Path("weights/voltage_model.pt")

torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Load the measurements

The last 20% of the shuffled rows is split evenly between validation and testing.
The validation set picks the checkpoint and the test set is used at the end.

In [ ]:
if not DATA.is_file():
    raise FileNotFoundError(f"Put your measured CSV at {DATA} or change DATA above.")

frame = pd.read_csv(DATA)
if len(frame) < 10 or frame.shape[1] <= SPECTRUM_POINTS:
    raise ValueError("Need at least 10 rows and at least one voltage column after the spectrum.")

values = frame.to_numpy(dtype=np.float32)
if not np.isfinite(values).all():
    raise ValueError("The spectrum and voltage columns must contain finite numbers.")

spectra = values[:, :SPECTRUM_POINTS]
voltages = values[:, SPECTRUM_POINTS:]
voltage_columns = [str(column) for column in frame.columns[SPECTRUM_POINTS:]]
train_idx, val_idx, test_idx = np.split(
    rng.permutation(len(frame)), [int(0.8 * len(frame)), int(0.9 * len(frame))]
)
train_spectra = spectra[train_idx]
spectrum_mean = float(train_spectra.mean())
spectrum_std = float(train_spectra.std())
if spectrum_std == 0:
    raise ValueError("The training spectra have zero variance.")

x = torch.from_numpy((spectra - spectrum_mean) / spectrum_std).unsqueeze(1)
y = torch.from_numpy(voltages)
train = DataLoader(TensorDataset(x[train_idx], y[train_idx]), BATCH_SIZE, shuffle=True)
val = DataLoader(TensorDataset(x[val_idx], y[val_idx]), BATCH_SIZE)
test = DataLoader(TensorDataset(x[test_idx], y[test_idx]), BATCH_SIZE)
print(f"{len(train_idx)} training, {len(val_idx)} validation, {len(test_idx)} test")
print("Voltage columns:", voltage_columns)

## Train and save

The Niu model has two convolution layers. The ConvNet has three and uses batch normalization.

In [ ]:
def build_model(name, spectrum_points, outputs):
    if name == "niu":
        features = nn.Sequential(
            nn.Conv1d(1, 16, 6), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(16, 32, 6), nn.ReLU(), nn.MaxPool1d(2),
        )
    elif name == "convnet":
        layers = []
        for inputs, channels in [(1, 16), (16, 32), (32, 64)]:
            layers.extend([
                nn.Conv1d(inputs, channels, 15, padding=7),
                nn.BatchNorm1d(channels), nn.ReLU(), nn.MaxPool1d(2),
            ])
        features = nn.Sequential(*layers)
    else:
        raise ValueError(f"Unknown model: {name}")

    with torch.no_grad():
        features.eval()
        size = features(torch.zeros(1, 1, spectrum_points)).numel()
    return nn.Sequential(features, nn.Flatten(), nn.Linear(size, 128), nn.ReLU(), nn.Linear(128, outputs))


@torch.no_grad()
def errors(model, batches):
    model.eval()
    squared_error = absolute_error = count = 0
    for inputs, labels in batches:
        error = model(inputs.to(device)) - labels.to(device)
        squared_error += error.square().sum().item()
        absolute_error += error.abs().sum().item()
        count += error.numel()
    return squared_error / count, absolute_error / count


model = build_model(MODEL, SPECTRUM_POINTS, len(voltage_columns)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

best_val = float("inf")
train_losses, val_losses = [], []
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0
    for inputs, labels in train:
        optimizer.zero_grad()
        loss = loss_fn(model(inputs.to(device)), labels.to(device))
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(inputs)
    train_losses.append(running_loss / len(train_idx))
    val_mse, _ = errors(model, val)
    val_losses.append(val_mse)

    if val_mse < best_val:
        best_val = val_mse
        best_weights = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    if epoch == 0 or (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1:3d}  train MSE {train_losses[-1]:.5f}  val MSE {val_mse:.5f}")

model.load_state_dict(best_weights)
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
torch.save({
    "weights": best_weights,
    "model": MODEL,
    "spectrum_points": SPECTRUM_POINTS,
    "spectrum_mean": spectrum_mean,
    "spectrum_std": spectrum_std,
    "voltage_columns": voltage_columns,
}, CHECKPOINT)

test_mse, test_mae = errors(model, test)
print(f"Test RMSE {np.sqrt(test_mse):.4f} V  MAE {test_mae:.4f} V")
print(f"Saved {CHECKPOINT}")

plt.plot(train_losses, label="train")
plt.plot(val_losses, label="validation")
plt.xlabel("Epoch")
plt.ylabel("MSE (V²)")
plt.legend()
plt.show()

## Predict from a spectrum

Replace the example trace with the desired spectrum on the training wavelength grid.
Pass the raw transmission values in the same units as the CSV.

In [ ]:
checkpoint = torch.load(CHECKPOINT, map_location="cpu", weights_only=True)
predictor = build_model(
    checkpoint["model"], checkpoint["spectrum_points"], len(checkpoint["voltage_columns"])
)
predictor.load_state_dict(checkpoint["weights"])
predictor.to(device).eval()


@torch.no_grad()
def predict_voltages(spectrum):
    spectrum = np.asarray(spectrum, dtype=np.float32)
    if spectrum.shape != (checkpoint["spectrum_points"],) or not np.isfinite(spectrum).all():
        raise ValueError("Pass one finite spectrum on the training grid.")
    scaled = (spectrum - checkpoint["spectrum_mean"]) / checkpoint["spectrum_std"]
    inputs = torch.from_numpy(scaled).view(1, 1, -1).to(device)
    return predictor(inputs).squeeze(0).cpu().numpy()


row = test_idx[0]
pd.DataFrame({
    "voltage": checkpoint["voltage_columns"],
    "applied_V": voltages[row],
    "predicted_V": predict_voltages(spectra[row]),
}).round(4)